### Tarea 1.1: configuración inicial y creación de agentes básicos

Configure un prototipo de agente de atención al cliente mediante el marco de Strands Agents. Este prototipo sirve como punto de partida para explorar todo el recorrido, desde un prototipo de agente hasta soluciones listas para producción.

Después de completar esta tarea, su agente tendrá la siguiente arquitectura básica:

<div style="text-align:left">
    <img src="images/architecture_lab1_strands_es_la.png" width="75%"/>
</div>

*Descripción de la imagen: prototipo de agente simple que se ejecuta localmente con herramientas locales*

Instale las dependencias e importe todas las bibliotecas necesarias, lo que incluye AWS SDK, los componentes de AgentCore y el marco Strands, para preparar el entorno de desarrollo.

In [ ]:
import boto3
import json
import uuid
import time
import requests
from boto3.session import Session

# AgentCore imports
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

# Strands imports
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

# Local tools
from lab_helpers.lab1_strands_agent import (
    get_product_info, get_return_policy, get_technical_support, web_search,
    SYSTEM_PROMPT, MODEL_ID
)
from lab_helpers.utils import get_ssm_parameter, put_ssm_parameter
from scripts.utils import get_cognito_client_secret

# Setup
boto_session = Session()
REGION = "us-east-1"
CUSTOMER_ID = "customer_001"
SESSION_ID = str(uuid.uuid4())

print("✅ Libraries imported successfully!")

Antes de crear el agente, revise las herramientas locales que impulsarán nuestras capacidades de atención al cliente. Abra y revise `lab_helpers/lab1_strands_agent.py` para comprender:

- Las herramientas se definen localmente en este archivo mediante el `@tool` decorador
- Las cuatro funciones de herramientas y sus propósitos:
  - get_product_info (): obtener información del producto
  - get_return_policy (): obtener la política de devolución de un producto específico
  - get_technical_support (): brindar orientación de soporte técnico
  - web_search (): explorar la web en busca de información actualizada
- Cómo utilizan los datos de simulación (simulación de bases de datos o API reales)
- La petición del sistema que determina el comportamiento del agente

Cree un agente de atención al cliente básico que exhiba las principales capacidades de la IA, desde la comprensión de consultas hasta la ejecución de acciones. Este agente combina lo siguiente:

- **Modelo fundacional**: el “cerebro” que impulsa el razonamiento y la toma de decisiones
- **Petición del sistema**: instrucciones de comportamiento que determinan la personalidad y los estándares de servicio del agente
- **Herramientas especializadas**: las cuatro herramientas locales (información del producto, política de devolución, soporte técnico y búsqueda web)

Cuando llama al agente, este sigue el siguiente proceso:
1. **Análisis de consultas**: el agente analiza la pregunta del cliente
2. **Selección de herramientas**: el agente determina qué herramientas utilizar (si las hay)
3. **Ejecución de herramientas**: el agente llama a la herramienta adecuada con los parámetros correctos
4. **Síntesis de la respuesta**: el agente combina los resultados de la herramienta con sus conocimientos para generar una respuesta útil
5. **Control de calidad**: el agente se asegura de que la respuesta cumpla con los estándares de la petición del sistema

In [ ]:
# Create a basic agent with local tools
model = BedrockModel(model_id=MODEL_ID, temperature=0.3, region_name=REGION)
basic_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Basic customer support agent ready!")
print("📋 Available tools: Product Info, Return Policy, Technical Support, Web Search")

Ponga a prueba a su agente básico para ver cómo maneja las consultas del cliente y cómo utiliza las herramientas:

In [ ]:
# Test basic agent functionality
print("💬 Testing basic agent...\n")
response = basic_agent("What's the return policy for laptops?")
print("\n" + "="*50 + "\n")

Este prototipo inicial presenta varias limitaciones que se abordarán en tareas posteriores:

- **Sin memoria persistente**: el agente olvida el historial del cliente y las preferencias de las sesiones anteriores
- **Solo herramientas locales**: sin integración con herramientas compartidas o de nivel empresarial  
- **Sin administración de identidades**: no puede actuar en nombre de usuarios específicos

### Tarea 1.2: mejorar el agente con memoria

Un cliente importante se comunica con el equipo de soporte por un problema con su último pedido. Explica sus preferencias, expresa su frustración y trabaja con el agente para solucionar el problema. Tres semanas después, se vuelve a comunicar con el servicio de soporte con una pregunta relacionada. Pero ahora debe repetirlo todo (sus preferencias, historial y contexto) porque el agente solo recuerda la sesión de conversación actual y no las sesiones anteriores. Esto provoca lo siguiente:
- **Clientes frustrados** que deben repetir su información una y otra vez
- **Soporte ineficiente** que no puede basarse en interacciones previas
- **Baja satisfacción del cliente** debido a respuestas genéricas e impersonales

Para abordar esta limitación, Amazon Bedrock AgentCore Memory proporciona un servicio administrado que permite a los agentes de IA mantener el contexto a lo largo del tiempo, recordar datos importantes y ofrecer experiencias personalizadas y consistentes. AgentCore Memory funciona en dos niveles:
- **Memoria a corto plazo**: contexto de conversación inmediata e información basada en sesiones (administrada automáticamente por el marco de Strands Agent)
- **Memoria a largo plazo**: información persistente extraída de varias conversaciones, que incluye hechos, preferencias y resúmenes (implementada mediante el servicio AgentCore Memory con las estrategias USER_PREFERENCE y SEMANTIC)

Convierta su prototipo en un asistente orientado al cliente que pueda realizar estas acciones:
- **“¡Bienvenida de nuevo, Sarah!”**: reconocer al instante a los clientes que regresan
- **“En relación con el problema de su portátil del mes pasado”**: conectar las conversaciones relacionadas sin problemas
- **“Según su historial de compras, esto es lo que recomiendo”**: brindar sugerencias personalizadas

Después de completar esta tarea, su agente tendrá la siguiente arquitectura con capacidades de memoria integradas:

<div style="text-align:left">
    <img src="images/architecture_lab2_memory_es_la.png" width="75%"/>
</div>

*Descripción de la imagen: agente mejorado con AgentCore Memory para mantener contexto persistente del cliente y ofrecer personalización*

**Configuración de estrategias de memoria**. Cree un recurso de memoria que combine dos estrategias inteligentes:

| Tipo de estrategia | Propósito | Beneficio para el cliente |
|---------------|---------|------------------|
| USER_PREFERENCE | Preferencias y comportamientos del cliente | “Recuerdo que prefieres…” |
| SEMANTIC | Información basada en hechos y contexto | “Con respecto a tu problema anterior…” |

AgentCore Memory usa espacios de nombres para agrupar de manera lógica los mensajes de memoria a largo plazo mediante el actorId:
- `support/customer/{actorId}/preferences`: para la estrategia de memoria de preferencias del usuario
- `support/customer/{actorId}/semantic`: para la estrategia de memoria semántica

In [ ]:
# Initialize memory client for AgentCore Memory service
memory_client = MemoryClient(region_name=REGION)
memory_name = "CustomerSupportMemory"

def create_or_get_memory_resource():
    try:
        # Try to get existing memory resource from SSM parameter
        memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
        memory_client.gmcp_client.get_memory(memoryId=memory_id)
        return memory_id
    except:
        # Create new memory resource with two strategies
        strategies = [
            {
                # USER_PREFERENCE strategy captures customer preferences and behaviors
                StrategyType.USER_PREFERENCE.value: {
                    "name": "CustomerPreferences",
                    "description": "Captures customer preferences and behavior",
                    "namespaces": ["support/customer/{actorId}/preferences"],
                }
            },
            {
                # SEMANTIC strategy stores factual information from conversations
                StrategyType.SEMANTIC.value: {
                    "name": "CustomerSupportSemantic",
                    "description": "Stores facts from conversations",
                    "namespaces": ["support/customer/{actorId}/semantic"],
                }
            },
        ]
        print("Creating AgentCore Memory resources (2-3 minutes)...")
        # Create memory resource and wait for completion
        response = memory_client.create_memory_and_wait(
            name=memory_name,
            description="Customer support agent memory",
            strategies=strategies,
            event_expiry_days=90,  # Memory events expire after 90 days
        )
        memory_id = response["id"]
        # Store memory ID in SSM for future use
        put_ssm_parameter("/app/customersupport/agentcore/memory_id", memory_id)
        return memory_id

memory_id = create_or_get_memory_resource()
print(f"✅ Memory resource ready: {memory_id}")

Simule a un cliente recurrente, llamado “customer_001”, que tuvo interacciones previas con su equipo de soporte. Esto demuestra cómo AgentCore Memory transforma automáticamente las conversaciones individuales en información valiosa y persistente sobre el cliente. Cargue las interacciones previas con el cliente y observe cómo AgentCore Memory las convierte automáticamente en información del cliente a largo plazo.

In [ ]:
# Seed previous customer interactions
previous_interactions = [
    ("I'm having issues with my MacBook Pro overheating during video editing.", "USER"),
    ("I can help with that thermal issue. Your MacBook Pro order #MB-78432 is still under warranty.", "ASSISTANT"),
    ("What's the return policy on gaming headphones? I need low latency for competitive FPS games", "USER"),
    ("For gaming headphones, you have 30 days to return. Since you're into competitive FPS, I'd recommend checking audio latency specs.", "ASSISTANT"),
    ("I need a laptop under $1200 for programming. Prefer 16GB RAM minimum and good Linux compatibility. I like ThinkPad models.", "USER"),
    ("Perfect! For development work, I'd suggest ThinkPad E series or Dell XPS models with excellent Linux support.", "ASSISTANT"),
]

if memory_id:
    memory_client.create_event(
        memory_id=memory_id,
        actor_id=CUSTOMER_ID,
        session_id="previous_session",
        messages=previous_interactions
    )
    print("✅ Customer history seeded successfully")
    print("⏳ Long-term memory processing will begin automatically...")

Strands Agents brinda un potente sistema de enlaces que permite a los componentes reaccionar al comportamiento del agente o modificarlo mediante funciones de devolución de llamada de eventos con tipos fuertemente definidos. Esto garantiza que las operaciones de memoria se realicen automáticamente sin intervención manual.

Cada vez que un cliente interactúe con el agente, este hará lo siguiente de forma automática:
- **Personalizar la conversación** en función de las interacciones y preferencias anteriores
- **Agregar interacciones nuevas a la memoria** para mejorar continuamente la personalización futura

Qué hace la integración de enlaces:
- **Antes de responder**: recupera automáticamente el contexto y las preferencias relevantes del cliente
- **Después de responder**: guarda de forma automática la interacción nueva en AgentCore Memory

Habilite el contexto automático del cliente con enlaces de memoria.

In [ ]:
class CustomerSupportMemoryHooks(HookProvider):
    def __init__(self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_customer_context(self, event: MessageAddedEvent):
        # Hook that runs before agent responds to retrieve customer context
        messages = event.agent.messages
        # Only process user messages (not tool results)
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                all_context = []
                # Retrieve memories from each strategy namespace, both USER_PREFERENCE and SEMANTIC
                for context_type, namespace in self.namespaces.items():
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id),
                        query=user_query,
                        top_k=3,  # Get top 3 relevant memories
                    )
                    # Extract text content from memory objects
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(f"[{context_type.upper()}] {text}")
                
                # Prepend customer context to user message
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0]["text"] = f"Customer Context:\n{context_text}\n\n{original_text}"
            except Exception as e:
                print(f"Failed to retrieve customer context: {e}")

    def save_support_interaction(self, event: AfterInvocationEvent):
        # Hook that runs after agent responds to save interaction to memory
        try:
            messages = event.agent.messages
            # Only save if we have both user and assistant messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                customer_query = None
                agent_response = None
                
                # Find the most recent user query and assistant response
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not customer_query and "toolResult" not in msg["content"][0]:
                        customer_query = msg["content"][0]["text"]
                        break
                
                # Save the interaction to AgentCore Memory
                if customer_query and agent_response:
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[(customer_query, "USER"), (agent_response, "ASSISTANT")],
                    )
        except Exception as e:
            print(f"Failed to save support interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        # Register both hooks with the agent's hook registry
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)

print("✅ Memory hooks defined - Automatic customer personalization enabled!")
print("🧠 Your agent will now remember customers and personalize every interaction")

Cree y pruebe el agente con memoria mejorada para ver cómo recupera el contexto del cliente y personaliza las respuestas:

In [ ]:
# Create memory-enhanced agent with hooks
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

memory_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    hooks=[memory_hooks],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Memory-enhanced agent created!")
print("🧠 Agent will automatically retrieve customer context and save interactions")

In [ ]:
# Wait for memory processing to complete
print("⏳ Waiting 90 seconds for memory processing to complete...")
time.sleep(90)

# Test memory recall
print("🧠 Testing memory-enhanced agent...\n")
response = memory_agent("What are my laptop preferences?")
print("\n" + "="*50 + "\n")

### Tarea 1.3: escalar mediante la integración de Gateway y AgentCore Identity

Con la memoria en funcionamiento, concéntrese en herramientas potentes y amplíe su impacto. Los mejores agentes requieren herramientas que aprovechen al máximo las API y los datos propios y de terceros, lo que les permite realizar tareas para clientes internos y externos. Sin embargo, crear, proteger y escalar las herramientas de agentes es difícil, lo que se convierte en un obstáculo significativo para que los clientes pasen de prototipos de agentes a obtener un valor empresarial real con agentes en producción. AgentCore Gateway funciona como una capa de conectividad que permite a los agentes de IA descubrir, autenticar e invocar herramientas del mundo real mediante un punto de conexión unificado del **protocolo de contexto para modelos (MCP)**. Esto es vital para las empresas que administran cientos de API, recursos y herramientas.

Beneficios clave:
- **Solución de servidor MCP completamente administrada** sin administración de infraestructuras
- **Integración de API** y funciones de Lambda existentes
- **Interfaz uniforme** en diversas herramientas
- **Autenticación y autorización seguras**
- **Descubrimiento y selección de herramientas semánticas**

##### Lo que creará:

**Centralización y reutilización de herramientas:**
- Migre la búsqueda web de una herramienta local a una AgentCore Gateway centralizada
- Integre las funciones de Lambda empresariales existentes (verificación de garantía)
- Cree una infraestructura de herramientas compartida a la que puedan acceder varios tipos de agentes

**Seguridad de nivel empresarial:**
- Implemente la autenticación basada en JWT con la integración de Cognito
- Configure una autorización entrante segura para el acceso a la puerta de enlace
- Establezca un control de acceso basado en identidades para el uso de herramientas

Esto crea una base escalable en la que las herramientas se administran de forma centralizada y se reutilizan en varios tipos de agentes, lo que elimina la duplicación de código y simplifica el mantenimiento.

AgentCore Identity también participa en este proceso. Permite que los agentes de IA accedan de forma segura a los recursos de AWS y ayuden a administrar la autenticación de llamadas entrantes al trabajar con Amazon Cognito. También permite que estos agentes accedan de forma segura a herramientas y servicios de terceros mediante la autenticación saliente, aunque esta característica de AgentCore Identity no se utiliza en este laboratorio.

<div style="text-align:left">
    <img src="images/architecture_lab3_identity_es_la.png" width="75%"/>
</div>

Después de completar esta tarea, su agente tendrá la siguiente arquitectura con capacidades de puerta de enlace integradas:

<div style="text-align:left">
    <img src="images/architecture_lab3_gateway_es_la.png" width="75%"/>
</div>

*Descripción de la imagen: agente mejorado con una AgentCore Gateway para garantizar una administración de herramientas segura y centralizada, con integración empresarial*

Cree la AgentCore Gateway para exponer una función de Lambda como un punto de conexión compatible con MCP. Para validar a las personas que llaman y están autorizadas a invocar sus herramientas, configure la **autenticación entrante** mediante la autorización OAuth, el estándar para los servidores MCP.

### Descripción de la autenticación de Gateway

AgentCore Gateway utiliza **OAuth 2.0 con tokens JWT** para proteger el acceso a sus herramientas. Esto evita que las aplicaciones no autorizadas invoquen sus funciones de Lambda.

**Conceptos clave:**

1. **Proveedor de autenticación**: Amazon Cognito administra las identidades y emite tokens
2. **Credenciales de cliente**: su agente usa un client_id y un client_secret (como un nombre de usuario y una contraseña para las aplicaciones)
3. **Tokens de JWT**: tokens de corta duración que confirman que el agente está autorizado
4. **Clientes permitidos**: una lista de identificadores de clientes autorizados que pueden acceder a la Gateway

**Cómo funciona:**
```
Agente → Cognito: “Estos son mi client_id y client_secret”
Cognito → Agente: “Este es su token de acceso de JWT”
Agente → Gateway: “Aquí está mi token”
Gateway → Cognito: “¿Este token es válido y corresponde a un cliente autorizado?”
Gateway → Agent: “Acceso concedido”
```

**Nota de seguridad**: Estas credenciales se crearon previamente para usted y se almacenaron de forma segura en el Almacén de parámetros de SSM. Las credenciales nunca deben codificarse directamente en el código

In [ ]:
# Retrieve authentication configuration from SSM Parameter Store
# These values were created by the CloudFormation template

# Client ID: Identifies which application is making the request
machine_client_id = get_ssm_parameter("/app/customersupport/agentcore/machine_client_id")
print(f"Machine Client ID: {machine_client_id}")

# Discovery URL: Tells the Gateway where to find Cognito's OAuth configuration
# This URL provides metadata about token endpoints, supported scopes, etc.
cognito_discovery_url = get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
print(f"Discovery URL: {cognito_discovery_url}")

# Configure JWT-based authentication for the Gateway
auth_config = {
    "customJWTAuthorizer": {
        # Only tokens from this client ID will be accepted
        "allowedClients": [machine_client_id],
        # Gateway will fetch OAuth metadata from this URL
        "discoveryUrl": cognito_discovery_url
    }
}

print("✅ Authentication configuration ready")

### Creación de la AgentCore Gateway

La Gateway actúa como un proxy seguro entre su agente y la función de Lambda backend. Piense en ello como una puerta de enlace de API diseñada específicamente para agentes de IA.

**Lo que creamos:**
- **Infraestructura de la Gateway**: el recurso principal de la Gateway
- **Protocolo MCP**: protocolo estándar para la comunicación de herramientas
- **Autorización JWT**: uso de la configuración de autenticación que acabamos de crear
- **Rol de IAM**: permisos para invocar la función de Lambda

**Qué ocurre luego:**
1. Cree la Gateway (esta celda)
2. Agregue un objetivo de Lambda con definiciones de herramientas (celda siguiente)
   - Una función de Lambda controla varias herramientas: `check_warranty_status` y `web_search`
   - La Gateway envía el nombre de la herramienta a Lambda, que lo dirige al controlador apropiado
3. Conecte su agente a Gateway

**Nota**: La plantilla de CloudFormation implementó una única función de Lambda (`CustomerSupportLambda`) que puede controlar varias operaciones de herramientas. Esto es más eficiente que implementar funciones de Lambda independientes para cada herramienta.

In [ ]:
class CreationFailedError(Exception):
    def __init__(self, message):
        self.message = message
        super().__init__(self.message)

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
gateway_name = "customersupport-gw"

try:
    print(f"Creating gateway: {gateway_name}")
    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn=get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role"),
        protocolType="MCP",  # Model Context Protocol
        authorizerType="CUSTOM_JWT",  # Use JWT tokens for auth
        authorizerConfiguration=auth_config,  # Our auth config from above
        description="Customer Support AgentCore Gateway",
    )
    gateway_id = create_response["gatewayId"]
    gateway_url = create_response["gatewayUrl"]
    put_ssm_parameter("/app/customersupport/agentcore/gateway_id", gateway_id)

    # Wait for Gateway to be ready
    print("Waiting for Gateway to be ready...")
    while True:
        status = gateway_client.get_gateway(gatewayIdentifier=gateway_id)['status']
        if status == 'READY':
            break
        elif status == 'FAILED':
            raise CreationFailedError("Gateway creation failed")
        else:
            print(f"  Status: {status}")
            time.sleep(5)

    print(f"✅ Gateway created successfully!")
    print(f"   Gateway ID: {gateway_id}")
    print(f"   Gateway URL: {gateway_url}")
    
except gateway_client.exceptions.ConflictException:
    # Gateway already exists, retrieve it
    gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
    gateway_url = gateway_response["gatewayUrl"]
    print(f"✅ Using existing gateway: {gateway_id}")
    
except CreationFailedError:
    print("\033[31m❌ Gateway creation failed. Check CloudWatch logs for details.\033[0m")

AgentCore Gateway llena el contexto de Lambda con el nombre de la herramienta que se invocará, mientras que los parámetros que se pasan a la herramienta se incluyen en el evento de Lambda. Esto le permite integrar las funciones de Lambda empresariales existentes (en este caso, `AgentCoreLab-CustomerSupportLambda`) que se pueden reutilizar en varios agentes.

Agregue sus funciones de Lambda como destinos de puerta de enlace mediante la especificación de la API:

In [ ]:
# Load API specification for Lambda tools
api_spec = [
    {
        "name": "check_warranty_status",
        "description": "Check warranty status using serial number and email",
        "inputSchema": {
            "type": "object",
            "properties": {
                "serial_number": {"type": "string"},
                "customer_email": {"type": "string"}
            },
            "required": ["serial_number"]
        }
    },
    {
        "name": "web_search",
        "description": "Search the web for updated information",
        "inputSchema": {
            "type": "object",
            "properties": {
                "keywords": {"type": "string", "description": "Search query keywords"},
                "region": {"type": "string", "description": "Search region (e.g., us-en)"},
                "max_results": {"type": "integer", "description": "Maximum results"}
            },
            "required": ["keywords"]
        }
    }
]

# Create gateway target
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": get_ssm_parameter("/app/customersupport/agentcore/lambda_arn"),
            "toolSchema": {"inlinePayload": api_spec},
        }
    }
}

try:
    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="LambdaTarget",
        description="Lambda tools for customer support",
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
    )
    print(f"✅ Gateway target created: {create_target_response['targetId']}")
except Exception as e:
    print(f"Gateway target may already exist: {str(e)}")

Integre el token de autenticación de Cognito en un cliente MCP del SDK de Strands para crear una conexión MCP segura.

Cree un cliente MCP autenticado para acceder a las herramientas de puerta de enlace:

In [ ]:
def get_cognito_client_secret():
    # Get Cognito client secret using Cognito API
    client = boto3.client("cognito-idp")
    response = client.describe_user_pool_client(
        UserPoolId=get_ssm_parameter("/app/customersupport/agentcore/userpool_id"),
        ClientId=get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
    )
    return response["UserPoolClient"]["ClientSecret"]

def get_oauth_token():
    # Get OAuth token for gateway authentication using client credentials flow
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    data = {
        "grant_type": "client_credentials",  # OAuth 2.0 client credentials flow
        "client_id": get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
        "client_secret": get_cognito_client_secret(),
        "scope": get_ssm_parameter("/app/customersupport/agentcore/cognito_auth_scope"),
    }
    # Request access token from Cognito
    response = requests.post(
        get_ssm_parameter("/app/customersupport/agentcore/cognito_token_url"),
        headers=headers, data=data
    )
    return response.json()

# Get OAuth access token (JWT format)
token_response = get_oauth_token()
access_token = token_response['access_token']

# Create MCP client with Bearer token authentication
mcp_client = MCPClient(
    url=gateway_url,
    headers={"Authorization": f"Bearer {access_token}"},  # JWT token in Authorization header
)

print(f"✅ MCP client configured for gateway: {gateway_url}")

Combine todo: enlaces de memoria, herramientas locales y herramientas de puerta de enlace. Esto crea una arquitectura híbrida en la que algunas herramientas permanecen locales para mayor velocidad y simplicidad, mientras que otras se centralizan mediante la puerta de enlace para facilitar su reutilización e integración empresarial.

Este enfoque elimina la duplicación de código en diferentes agentes y crea una administración centralizada para las actualizaciones de las herramientas:

In [ ]:
# Initialize memory hooks for customer context
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

# Start MCP client connection to gateway
mcp_client.start()
# Retrieve available tools from the gateway
gateway_tools = mcp_client.list_tools_sync()

# Combine local tools with centralized gateway tools
all_tools = [
    get_product_info,      # Local tool
    get_return_policy,     # Local tool
    get_technical_support, # Local tool
] + gateway_tools          # Gateway tools - web_search, check_warranty_status

# Create enhanced agent with memory and gateway integration
enhanced_agent = Agent(
    model=model,
    tools=all_tools,           # Local + gateway tools
    hooks=[memory_hooks],      # Automatic memory operations
    system_prompt=SYSTEM_PROMPT
)

print("✅ Enhanced Customer Support Agent created!")
print(f"📊 Total tools available: {len(all_tools)}")
print(f"🧠 Memory enabled with ID: {memory_id}")
print(f"🔒 Secure gateway integration: {gateway_url}")

Pruebe su agente con capacidades de memoria y puerta de enlace. Verifique que el agente pueda utilizar sin problemas tanto las herramientas locales como las herramientas centralizadas de la puerta de enlace, al tiempo que mantiene el contexto del cliente mediante la memoria.

Los escenarios de prueba incluyen la verificación de garantía, la búsqueda web y las capacidades combinadas de memoria y puerta de enlace:

In [ ]:
# Test gateway tools
print("🔍 Testing gateway web search...\n")
response2 = enhanced_agent("Search for latest iPhone 15 troubleshooting tips")
print("\n" + "="*50 + "\n")

In [ ]:
# Test warranty check
print("🛡️ Testing warranty check...\n")
response3 = enhanced_agent("Check warranty status for serial number ABC12345678")
print("\n" + "="*50 + "\n")

In [ ]:
# Test combined capabilities
print("🎯 Testing combined memory + gateway capabilities...\n")
response4 = enhanced_agent("I need gaming headphones again, and also search for the latest reviews")
print("\n" + "="*50 + "\n")

## Próximos pasos

🎉 **¡Felicitaciones!** Ha completado los ejercicios del cuaderno.

Llevó a cabo correctamente las siguientes tareas:
- Crear un prototipo de agente de IA básico mediante Strands
- Lo mejoró con AgentCore Memory para garantizar un contexto de cliente persistente
- Lo integró con AgentCore Gateway para permitir el uso compartido centralizado y seguro de herramientas
- Probó el sistema completo de atención al cliente listo para la producción

### ¿Cuál es el siguiente paso?

1. **Cierre el archivo de este cuaderno**
2. **Regrese a las instrucciones del laboratorio**
3. **Continúe con la Tarea 2** para explorar el panel de AgentCore y ver sus recursos en acción
